In [ ]:
!pip install -q -U langchain langchain-groq langchain-core langsmith

In [ ]:
import os

key = input("Paste your Groq API key: ")
os.environ["GROQ_API_KEY"] = key
os.environ["LANGCHAIN_TRACING_V2"] = "false"
print("Done!")

In [ ]:
job_description = """
Job Title: Data Scientist
Required Skills: Python, Machine Learning, SQL, TensorFlow, Data Analysis, Statistics
Experience: 3+ years
Tools: Pandas, NumPy, Scikit-learn, Git
Education: Bachelor's or Master's in Computer Science, Statistics, or related field
"""

resume_strong = """
Name: Ayesha Raza
Experience: 5 years as Data Scientist at Google and Amazon
Skills: Python, Machine Learning, Deep Learning, SQL, Statistics, Data Analysis, NLP
Tools: TensorFlow, PyTorch, Scikit-learn, Pandas, NumPy, Git, Tableau
Education: M.Tech in Computer Science, IIT Hyderabad
Projects: Built a recommendation engine that increased CTR by 35%. Led ML model deployment pipelines.
"""

resume_average = """
Name: Rahul Mehta
Experience: 2 years as Junior Data Analyst at a startup
Skills: Python, SQL, Data Analysis, Basic Machine Learning
Tools: Pandas, NumPy, Excel, Git
Education: B.Sc in Statistics, Osmania University
Projects: Created dashboards using Python. Wrote SQL queries for business reports.
"""

resume_weak = """
Name: Sanjay Gupta
Experience: 6 months internship in web development
Skills: HTML, CSS, JavaScript, basic Python
Tools: VS Code, GitHub
Education: B.Com, pursuing online Python course
Projects: Built a personal portfolio website.
"""

resumes = {
    "Strong Candidate (Ayesha)": resume_strong,
    "Average Candidate (Rahul)": resume_average,
    "Weak Candidate (Sanjay)": resume_weak,
}

In [ ]:
from langchain_groq import ChatGroq
from langchain_core.output_parsers import StrOutputParser
import time

llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)
parser = StrOutputParser()

extraction_chain  = extraction_prompt  | llm | parser
matching_chain    = matching_prompt    | llm | parser
scoring_chain     = scoring_prompt     | llm | parser
explanation_chain = explanation_prompt | llm | parser
print("LLM ready!")

In [ ]:
import json

def run_pipeline(candidate_name, resume, job_description):
    print(f"\n{'='*60}")
    print(f"  Processing: {candidate_name}")
    print(f"{'='*60}")

    extracted = extraction_chain.invoke({"resume": resume})
    print(f"\n[Step 1] Extracted Info:\n{extracted}")

    matched = matching_chain.invoke({
        "extracted_info": extracted,
        "job_description": job_description
    })
    print(f"\n[Step 2] Match Result:\n{matched}")

    scored = scoring_chain.invoke({
        "match_result": matched,
        "job_description": job_description
    })
    print(f"\n[Step 3] Score:\n{scored}")

    explanation = explanation_chain.invoke({
        "score_result": scored,
        "match_result": matched,
        "candidate_name": candidate_name
    })
    print(f"\n[Step 4] Explanation:\n{explanation}")

    try:
        clean = scored.strip().replace("```json", "").replace("```", "").strip()
        score_data = json.loads(clean)
        final_score = score_data.get("score", "N/A")
    except json.JSONDecodeError:
        final_score = "Parse Error"

    print(f"\n>>> FINAL SCORE for {candidate_name}: {final_score}/100")
    print(f"{'='*60}\n")

    return {
        "candidate": candidate_name,
        "extracted": extracted,
        "matched": matched,
        "scored": scored,
        "final_score": final_score,
        "explanation": explanation
    }

In [ ]:
results = []

for name, resume in resumes.items():
    result = run_pipeline(name, resume, job_description)
    results.append(result)

In [ ]:
import pandas as pd

summary = []
for r in results:
    try:
        clean = r["scored"].strip().replace("```json", "").replace("```", "").strip()
        score_data = json.loads(clean)
        summary.append({
            "Candidate": r["candidate"],
            "Final Score": score_data.get("score", "N/A"),
            "Skills (40)": score_data.get("skills_score", "N/A"),
            "Tools (20)": score_data.get("tools_score", "N/A"),
            "Experience (25)": score_data.get("experience_score", "N/A"),
            "Education (15)": score_data.get("education_score", "N/A"),
        })
    except Exception:
        summary.append({"Candidate": r["candidate"], "Final Score": "Error"})

df = pd.DataFrame(summary)
print("\n===== RESUME SCREENING SUMMARY =====")
print(df.to_string(index=False))

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableConfig

few_shot_scoring_prompt = PromptTemplate(
    input_variables=["match_result", "job_description"],
    template="""
You are a hiring evaluator. Score candidates from 0-100.

Examples:
- All skills matched, 5 years experience, relevant tools: Score = 92
- Half skills matched, 1 year experience, few tools: Score = 48
- No relevant skills, no experience: Score = 12

Now evaluate:

Match Analysis:
{match_result}

Job Description:
{job_description}

Return JSON:
{{
  "score": 0,
  "skills_score": 0,
  "tools_score": 0,
  "experience_score": 0,
  "education_score": 0
}}

Return ONLY the JSON. No markdown, no explanation.
"""
)

few_shot_chain = few_shot_scoring_prompt | llm | parser
config = RunnableConfig(tags=["few-shot", "bonus", "scoring"])

few_shot_result = few_shot_chain.invoke(
    {
        "match_result": results[0]["matched"],
        "job_description": job_description
    },
    config=config
)

print("Few-Shot Score (Strong Candidate):")
print(few_shot_result)

In [ ]:
print("="*50)
print("PIPELINE COMPLETE")
print("="*50)
print(f"Total Candidates Processed: {len(results)}")
for r in results:
    print(f"  - {r['candidate']}: {r['final_score']}/100")
print("\nLangSmith Tracing:", os.environ.get("LANGCHAIN_TRACING_V2"))
print("Project:", os.environ.get("LANGCHAIN_PROJECT"))
print("="*50)